# Level 4: Data Cleaning, Scientific Data Analysis, and Visualization

**Course:** ICS 2207 — Scientific Computing  
**Project:** HydroSense-Kenya  
**Objective:** Use Pandas and Matplotlib to transform raw scientific data into interpretable evidence.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.data_cleaning import (
    load_datasets, clean_weather, clean_soil, save_cleaned_dataset
)

weather_raw, soil_raw, params = load_datasets()

---
## 1. Identify Data Quality Issues

In [ ]:
print('=== WEATHER — Missing Values ===')
print(weather_raw.isnull().sum())
print('\nRows with missing data:')
weather_raw[weather_raw.isnull().any(axis=1)]

In [ ]:
print('=== WEATHER — Potential Outliers ===')
print(f"Temperature max: {weather_raw['temperature_c'].max()}°C (Mar 14 — likely sensor error)")
print(f"Rainfall max:    {weather_raw['rainfall_mm'].max()} mm (Mar 26 — extreme but plausible)")
print()
print('=== SOIL — Missing & Anomalies ===')
print(f"Missing soil_moisture: {soil_raw['soil_moisture_pct'].isna().sum()}")
print(f"Tank level max: {soil_raw['tank_level_liters'].max()} L (typical ~3400-4800)")
print(f"Pump flow min:  {soil_raw['pump_flow_lpm'].min()} LPM")
print(f"CHECK status rows: {(soil_raw['sensor_status'] == 'CHECK').sum()}")
print()
soil_raw[soil_raw['sensor_status'] == 'CHECK']

---
## 2. Clean Datasets (with documented decisions)

In [ ]:
weather_clean, weather_log = clean_weather(weather_raw)
soil_clean, soil_log = clean_soil(soil_raw)

print('=== Weather Cleaning Log ===')
for entry in weather_log:
    print(f'  • {entry}')

print('\n=== Soil Cleaning Log ===')
for entry in soil_log:
    print(f'  • {entry}')

In [ ]:
# Save cleaned dataset
merged, output_path = save_cleaned_dataset(weather_clean, soil_clean, params)
print(f'Cleaned dataset saved to {output_path}')
print(f'Shape: {merged.shape}')
print(f'Remaining NaN: {merged.isnull().sum().sum()}')

---
## 3. Descriptive Statistics

In [ ]:
print('=== Weather (cleaned) ===')
weather_clean.describe().round(2)

In [ ]:
print('=== Soil Sensor (cleaned) — by Zone ===')
soil_clean.groupby('zone_id')[['soil_moisture_pct','tank_level_liters','pump_flow_lpm','pump_power_watts']].describe().round(2).T

---
## 4. Scientific Visualizations

### Visualization 1: Rainfall Distribution with Cleaned Temperature Overlay

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(weather_clean['date'], weather_clean['rainfall_mm'],
        color='#3b82f6', alpha=0.8, label='Rainfall')
ax1.set_ylabel('Rainfall (mm)', color='#3b82f6', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#3b82f6')

ax2 = ax1.twinx()
ax2.plot(weather_clean['date'], weather_clean['temperature_c'],
         'o-', color='#ef4444', linewidth=1.5, markersize=4, label='Temperature')
ax2.set_ylabel('Temperature (°C)', color='#ef4444', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#ef4444')

ax1.set_title('Daily Rainfall and Temperature — March 2026 (Cleaned)',
              fontsize=14, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=3))
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95), fontsize=10)
ax1.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/level4_viz1_rainfall_temp.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Rainfall is episodic with dry spells (Mar 13-14, Mar 28-29) interspersed with wet days. The cleaned temperature series now shows a realistic range (21-27°C) after removing the 45.8°C outlier. Temperature tends to rise during dry spells and dip after heavy rain, consistent with evaporative cooling and cloud cover effects.

### Visualization 2: Soil Moisture Trends by Zone with Stress Thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = {'Zone_A': '#22c55e', 'Zone_B': '#f59e0b', 'Zone_C': '#8b5cf6'}
labels = {'Zone_A': 'Zone A (Tomato)', 'Zone_B': 'Zone B (Kale)', 'Zone_C': 'Zone C (Maize)'}

for zone_id in ['Zone_A', 'Zone_B', 'Zone_C']:
    zd = soil_clean[soil_clean['zone_id'] == zone_id]
    zp = params[params['zone_id'] == zone_id].iloc[0]
    ax.plot(zd['timestamp'], zd['soil_moisture_pct'], 'o-', markersize=3,
            linewidth=1.5, color=colors[zone_id], label=labels[zone_id])
    ax.axhline(zp['min_moisture_pct'], linestyle='--', color=colors[zone_id],
               alpha=0.5, linewidth=0.9)
    ax.axhline(zp['target_moisture_pct'], linestyle=':', color=colors[zone_id],
               alpha=0.4, linewidth=0.9)

ax.set_ylabel('Soil Moisture (%)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title('Soil Moisture Trends (dashed=min, dotted=target)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.savefig('../reports/level4_viz2_soil_moisture.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** All three zones show a declining moisture trend over March, indicating cumulative water loss exceeding inputs. Zone C (maize) drops below its minimum threshold earliest, suggesting it is the most drought-vulnerable zone — consistent with its larger area (180 m²) and higher drainage coefficient (0.22). The Mar 26 rainfall event temporarily reverses the decline across all zones.

### Visualization 3: Tank Level and Pump Flow Correlation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle('Tank Level vs. Pump Flow by Zone', fontsize=14, fontweight='bold')

for idx, zone_id in enumerate(['Zone_A', 'Zone_B', 'Zone_C']):
    zd = soil_clean[soil_clean['zone_id'] == zone_id]
    ax = axes[idx]
    sc = ax.scatter(zd['pump_flow_lpm'], zd['tank_level_liters'],
                    c=range(len(zd)), cmap='viridis', s=40, edgecolors='white', linewidth=0.5)
    ax.set_xlabel('Pump Flow (LPM)', fontsize=10)
    ax.set_title(labels[zone_id], fontsize=11)
    ax.grid(alpha=0.3)
    # Correlation
    r = zd['pump_flow_lpm'].corr(zd['tank_level_liters'])
    ax.annotate(f'r = {r:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[0].set_ylabel('Tank Level (litres)', fontsize=10)
plt.colorbar(sc, ax=axes[-1], label='Day index (early → late)', pad=0.02)
plt.tight_layout()
plt.savefig('../reports/level4_viz3_tank_pump.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Tank levels decline over time (colour gradient from early=purple to late=yellow) as water is consumed. The negative correlation between pump flow and tank level in Zones B and C suggests higher pumping rates draw down the tank faster. Zone A shows a weaker correlation, possibly because tomatoes require less intensive irrigation than kale or maize at this growth stage.

### Visualization 4: Pump Power vs. Pump Flow — Efficiency Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for zone_id in ['Zone_A', 'Zone_B', 'Zone_C']:
    zd = soil_clean[soil_clean['zone_id'] == zone_id]
    ax.scatter(zd['pump_flow_lpm'], zd['pump_power_watts'],
              s=50, alpha=0.7, color=colors[zone_id], label=labels[zone_id],
              edgecolors='white', linewidth=0.5)

# Overall trend line
all_flow = soil_clean['pump_flow_lpm'].values
all_power = soil_clean['pump_power_watts'].values
mask = ~(np.isnan(all_flow) | np.isnan(all_power))
z = np.polyfit(all_flow[mask], all_power[mask], 1)
x_line = np.linspace(all_flow[mask].min(), all_flow[mask].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'k--', linewidth=1.5,
        label=f'Trend: {z[0]:.1f}W per LPM')

ax.set_xlabel('Pump Flow (LPM)', fontsize=11)
ax.set_ylabel('Pump Power (Watts)', fontsize=11)
ax.set_title('Pump Power vs. Flow Rate — Energy Efficiency', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/level4_viz4_pump_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Pump power increases approximately linearly with flow rate, as expected for centrifugal pumps operating within their design range. Zone C consistently operates at higher flow rates and power, reflecting its larger area. The slope (~7-8 W per LPM) represents the marginal energy cost of additional pumping — a key input for the optimization model in Level 5.

### Visualization 5: Weather Correlation Heatmap

In [ ]:
weather_num = weather_clean[['rainfall_mm', 'temperature_c', 'humidity_pct',
                              'wind_speed_mps', 'solar_index']]
corr = weather_num.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

labels_short = ['Rainfall', 'Temp', 'Humidity', 'Wind', 'Solar']
ax.set_xticks(range(len(labels_short)))
ax.set_yticks(range(len(labels_short)))
ax.set_xticklabels(labels_short, fontsize=10)
ax.set_yticklabels(labels_short, fontsize=10)

for i in range(len(labels_short)):
    for j in range(len(labels_short)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')

plt.colorbar(im, label='Pearson Correlation', shrink=0.8)
ax.set_title('Weather Variable Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/level4_viz5_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Rainfall shows a negative correlation with temperature and solar index — rainy days are cooler and cloudier. Humidity and rainfall are positively correlated as expected. Wind speed is weakly correlated with other variables, behaving as a semi-independent forcing factor. These relationships validate the ET formula structure: higher temperature and solar radiation increase ET, while higher humidity reduces it.

### Visualization 6: Before vs. After Cleaning Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Data Cleaning — Before vs. After', fontsize=14, fontweight='bold')

# Temperature
ax = axes[0, 0]
ax.plot(weather_raw['date'], weather_raw['temperature_c'], 'o-',
        color='#ef4444', markersize=4, alpha=0.6, label='Raw')
ax.plot(weather_clean['date'], weather_clean['temperature_c'], 's-',
        color='#22c55e', markersize=4, label='Cleaned')
ax.set_title('Temperature', fontsize=11)
ax.set_ylabel('°C')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Tank level (Zone C)
ax = axes[0, 1]
zc_raw = soil_raw[soil_raw['zone_id'] == 'Zone_C']
zc_cln = soil_clean[soil_clean['zone_id'] == 'Zone_C']
ax.plot(zc_raw['timestamp'], zc_raw['tank_level_liters'], 'o-',
        color='#ef4444', markersize=4, alpha=0.6, label='Raw')
ax.plot(zc_cln['timestamp'], zc_cln['tank_level_liters'], 's-',
        color='#22c55e', markersize=4, label='Cleaned')
ax.set_title('Tank Level — Zone C', fontsize=11)
ax.set_ylabel('Litres')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Soil moisture (Zone B)
ax = axes[1, 0]
zb_raw = soil_raw[soil_raw['zone_id'] == 'Zone_B']
zb_cln = soil_clean[soil_clean['zone_id'] == 'Zone_B']
ax.plot(zb_raw['timestamp'], zb_raw['soil_moisture_pct'], 'o-',
        color='#ef4444', markersize=4, alpha=0.6, label='Raw')
ax.plot(zb_cln['timestamp'], zb_cln['soil_moisture_pct'], 's-',
        color='#22c55e', markersize=4, label='Cleaned')
ax.set_title('Soil Moisture — Zone B', fontsize=11)
ax.set_ylabel('%')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Pump flow (Zone B)
ax = axes[1, 1]
ax.plot(zb_raw['timestamp'], zb_raw['pump_flow_lpm'], 'o-',
        color='#ef4444', markersize=4, alpha=0.6, label='Raw')
ax.plot(zb_cln['timestamp'], zb_cln['pump_flow_lpm'], 's-',
        color='#22c55e', markersize=4, label='Cleaned')
ax.set_title('Pump Flow — Zone B', fontsize=11)
ax.set_ylabel('LPM')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/level4_viz6_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** The before/after comparison validates that our cleaning operations are targeted and conservative: only genuine anomalies are corrected (temperature spike, tank level spike, pump fault, moisture anomaly), while the overall data structure and trends remain intact. This is essential for scientific credibility — aggressive cleaning can introduce bias.

---
*End of Level 4 — Data Cleaning, Scientific Data Analysis, and Visualization.*